In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

print("=" * 70)
print("📊 NHANES VERİ ANALİZİ BAŞLADI!")
print("=" * 70)

# Dosyaları yükle
csv_files = {
    'demographic': 'data/raw/demographic.csv',
    'diet': 'data/raw/diet.csv',
    'examination': 'data/raw/examination.csv',
    'labs': 'data/raw/labs.csv',
    'medications': 'data/raw/medications.csv',
    'questionnaire': 'data/raw/questionnaire.csv'
}

dfs = {}
encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']

for name, path in csv_files.items():
    if os.path.exists(path):
        for encoding in encodings:
            try:
                dfs[name] = pd.read_csv(path, encoding=encoding)
                print(f"\n✅ {name.upper()} (Encoding: {encoding})")
                print(f"   Satır: {len(dfs[name]):,}")
                print(f"   Sütun: {len(dfs[name].columns)}")
                break
            except UnicodeDecodeError:
                continue
    else:
        print(f"❌ {name} bulunamadı!")

print("\n" + "=" * 70)
if 'demographic' in dfs:
    print("📋 DEMOGRAPHIC (Kişi Bilgileri):")
    print(dfs['demographic'].head(3))
    print(f"\nSütunlar ({len(dfs['demographic'].columns)}):")
    print(list(dfs['demographic'].columns[:10]))  # İlk 10 sütun

In [ ]:
# Labs dosyasının NAFLD ile ilgili kolonlarını ara
print("🔍 NAFLD GÖSTERGELERİ - LABS KOLONLARI")
print("=" * 70)

# Aradığımız kolon isimleri
nafld_keywords = ['AST', 'ALT', 'PLATELET', 'TRIGLY', 'HDL', 'LDL', 'GLUCOSE', 'ALBUMIN']

for keyword in nafld_keywords:
    matches = [col for col in dfs['labs'].columns if keyword in col.upper()]
    if matches:
        print(f"\n✅ {keyword}:")
        for col in matches:
            print(f"   • {col}")
    else:
        print(f"\n❓ {keyword}: Bulunamadı")

# İlk 20 kolon adını göster (full list çok uzun)
print("\n" + "=" * 70)
print("📋 LABS İlk 20 Kolon:")
for i, col in enumerate(dfs['labs'].columns[:20], 1):
    print(f"{i:2}. {col}")

In [ ]:
# Tüm LABS kolonlarını göster
print("📋 LABS TÜM 424 KOLON:")
print("=" * 70)

for i, col in enumerate(dfs['labs'].columns, 1):
    print(f"{i:3}. {col}")

In [ ]:
# NAFLD için gerekli kolonlar
print("🏥 NAFLD ÖZELLİKLERİ - KOLON EŞLEŞMESİ")
print("=" * 70)

nafld_columns = {
    'AST': 'LBXSASSI',              # Aspartate aminotransferase
    'ALT': 'LBXSATSI',              # Alanine aminotransferase
    'PLATELET': 'LBXPLTSI',         # Platelet count (1000/uL)
    'TRIGLYCERIDE': 'LBXTTG',       # Total triglyceride (mg/dL)
    'CHOLESTEROL': 'LBXTC',         # Total cholesterol (mg/dL)
    'LDL': 'LBDLDL',                # LDL cholesterol
}

print("\n✅ BULDUĞUMUZ KOLONLAR:\n")
for name, col in nafld_columns.items():
    if col in dfs['labs'].columns:
        print(f"   ✅ {name:15} → {col}")
    else:
        print(f"   ❌ {name:15} → {col} (BULUNAMADI)")

# HDL'yi ara
print("\n🔍 HDL aranıyor...")
hdl_matches = [col for col in dfs['labs'].columns if 'HDL' in col.upper()]
if hdl_matches:
    print(f"   ✅ HDL bulundu: {hdl_matches}")
else:
    print(f"   ❓ HDL bulunamadı, benzer kolonlar:")
    similar = [col for col in dfs['labs'].columns if 'H' in col and 'LB' in col]
    for col in similar[:10]:
        print(f"      • {col}")

In [ ]:
# Şüpheli HDL kolonlarını kontrol et
print("🔍 HDL ÖN AYIKLAMA:")
print("=" * 70)

suspicious_hdl = ['LBXSCH', 'LBXSGLSI', 'LBXTC']

for col in suspicious_hdl:
    if col in dfs['labs'].columns:
        print(f"\n📊 {col}:")
        print(f"   Non-null: {dfs['labs'][col].notna().sum():,}")
        print(f"   Mean: {dfs['labs'][col].mean():.2f}")
        print(f"   Min: {dfs['labs'][col].min():.2f}")
        print(f"   Max: {dfs['labs'][col].max():.2f}")
        print(f"   Sample: {dfs['labs'][col].dropna().head(3).values}")

# NHANES documentation'a göre HDL tipik değerleri:
# HDL: 20-100 mg/dL (düşük = risk)
# Total Cholesterol: 100-300 mg/dL
# LDL: 20-200 mg/dL

print("\n💡 İPUÇ: HDL tipik aralık 20-100 mg/dL")
print("    İyi HDL: >40 (erkek), >50 (kadın)")

In [ ]:
# HDL'yi hesapla
print("📊 HDL HESAPLAMA")
print("=" * 70)

# HDL = Total Cholesterol - LDL - (Triglyceride / 5)
dfs['labs']['HDL_calculated'] = (
    dfs['labs']['LBXTC'] - 
    dfs['labs']['LBDLDL'] - 
    (dfs['labs']['LBXTTG'] / 5)
)

print(f"\n✅ HDL Hesaplandı!")
print(f"\n📊 HDL İstatistikleri:")
print(f"   Non-null: {dfs['labs']['HDL_calculated'].notna().sum():,}")
print(f"   Mean: {dfs['labs']['HDL_calculated'].mean():.2f}")
print(f"   Min: {dfs['labs']['HDL_calculated'].min():.2f}")
print(f"   Max: {dfs['labs']['HDL_calculated'].max():.2f}")
print(f"   Sample: {dfs['labs']['HDL_calculated'].dropna().head(5).values}")

In [ ]:
# Examination'da BMI, BP ara
print("\n🏥 EXAMINATION KOLONLARı")
print("=" * 70 + "\n")

for i, col in enumerate(dfs['examination'].columns[:50], 1):
    print(f"{i:2}. {col}")
    

In [ ]:
# NAFLD MASTER DATAFRAME
print("🔗 NAFLD MASTER DATAFRAME OLUŞTURULUYOR...")
print("=" * 70)

# Demographic temel bilgiler
nafld_master = dfs['demographic'][['SEQN', 'RIDAGEYR', 'RIAGENDR']].copy()

# Rename
nafld_master = nafld_master.rename(columns={
    'RIDAGEYR': 'Age',
    'RIAGENDR': 'Gender'  # 1=Male, 2=Female
})

# Labs'tan NAFLD parametreleri
nafld_master = nafld_master.merge(
    dfs['labs'][['SEQN', 'LBXSASSI', 'LBXSATSI', 'LBXPLTSI', 'LBXTTG', 'LBXTC', 'LBDLDL', 'HDL_calculated']],
    on='SEQN',
    how='inner'
)

# Examination'dan fiziksel veriler
nafld_master = nafld_master.merge(
    dfs['examination'][['SEQN', 'BPXSY1', 'BPXDI1', 'BMXWT', 'BMXHT', 'BMXBMI', 'BMXWAIST']],
    on='SEQN',
    how='inner'
)

# Kolon adlarını düzelt
nafld_master = nafld_master.rename(columns={
    'LBXSASSI': 'AST',
    'LBXSATSI': 'ALT',
    'LBXPLTSI': 'Platelet_K',  # 1000/uL
    'LBXTTG': 'Triglyceride',   # mg/dL
    'LBXTC': 'Total_Cholesterol', # mg/dL
    'LBDLDL': 'LDL',            # mg/dL
    'HDL_calculated': 'HDL',    # mg/dL
    'BPXSY1': 'SBP',            # Systolic BP
    'BPXDI1': 'DBP',            # Diastolic BP
    'BMXWT': 'Weight_kg',
    'BMXHT': 'Height_cm',
    'BMXBMI': 'BMI',
    'BMXWAIST': 'Waist_cm'
})

print(f"✅ NAFLD Master DF Oluşturuldu!")
print(f"   Satır: {len(nafld_master):,}")
print(f"   Sütun: {len(nafld_master.columns)}")

print(f"\n📋 Sütunlar:")
print(list(nafld_master.columns))

print(f"\n📊 İlk 5 satır:")
print(nafld_master.head())

print(f"\n📈 Temel İstatistikler:")
print(nafld_master.describe())

print(f"\n🔴 Eksik Veriler:")
print(nafld_master.isnull().sum())

In [ ]:
# NAFLD Risk Skorları Hesapla
print("\n" + "=" * 70)
print("🏥 NAFLD RISK SKORLARI HESAPLANIYOR...")
print("=" * 70)

# FIB-4 Index = (Age × AST) / (Platelet × √ALT)
nafld_master['FIB4_Index'] = (
    (nafld_master['Age'] * nafld_master['AST']) / 
    (nafld_master['Platelet_K'] * np.sqrt(nafld_master['ALT']))
)

# APRI Score = ((AST/40) / Platelet) × 100
nafld_master['APRI_Score'] = (
    ((nafld_master['AST'] / 40) / nafld_master['Platelet_K']) * 100
)

# TG/HDL Ratio
nafld_master['TG_HDL_Ratio'] = nafld_master['Triglyceride'] / nafld_master['HDL']

print(f"\n✅ NAFLD Skorları Hesaplandı!")

print(f"\n📊 FIB-4 Index Dağılımı:")
print(f"   Mean: {nafld_master['FIB4_Index'].mean():.3f}")
print(f"   >1.3: {(nafld_master['FIB4_Index'] > 1.3).sum():,} kişi (Risk)")

print(f"\n📊 APRI Score Dağılımı:")
print(f"   Mean: {nafld_master['APRI_Score'].mean():.3f}")
print(f"   >1.0: {(nafld_master['APRI_Score'] > 1.0).sum():,} kişi (Risk)")

print(f"\n📊 TG/HDL Ratio Dağılımı:")
print(f"   Mean: {nafld_master['TG_HDL_Ratio'].mean():.3f}")
print(f"   >2.0: {(nafld_master['TG_HDL_Ratio'] > 2.0).sum():,} kişi (Insulin Resistance)")

print(f"\n📋 Güncellenmiş Master DF:")
print(nafld_master[['Age', 'Gender', 'FIB4_Index', 'APRI_Score', 'TG_HDL_Ratio']].head(10))

In [ ]:
# Veri temizleme
print("🧹 VERİ TEMİZLEME BAŞLANIYOR...")
print("=" * 70)

# Çalışma kopyası
nafld_clean = nafld_master.dropna(subset=['AST', 'ALT', 'Platelet_K'])

print(f"1️⃣  AST/ALT eksik olanlar kaldırıldı")
print(f"   Kalan: {len(nafld_clean):,} kişi ({len(nafld_clean)/len(nafld_master)*100:.1f}%)")

# BMI eksik olanlar kaldır
nafld_clean = nafld_clean.dropna(subset=['BMI', 'Weight_kg', 'Height_cm'])
print(f"\n2️⃣  BMI, Weight, Height eksik kaldırıldı")
print(f"   Kalan: {len(nafld_clean):,} kişi")

# Yaşı 18+ olanları filtrele (Yetişkin NAFLD)
nafld_clean = nafld_clean[nafld_clean['Age'] >= 18]
print(f"\n3️⃣  Yaş >= 18 filtrelemesi")
print(f"   Kalan: {len(nafld_clean):,} kişi (Yetişkin)")

# Triglyceride değerleri kontrol et
print(f"\n📊 Triglyceride Değerleri:")
print(f"   Min: {nafld_clean['Triglyceride'].min()}")
print(f"   Max: {nafld_clean['Triglyceride'].max()}")
print(f"   Mean: {nafld_clean['Triglyceride'].mean():.2f}")

print(f"\n✅ Temiz veri: {len(nafld_clean):,} kişi")

In [ ]:
# NAFLD Risk Kategorileri
print("\n" + "=" * 70)
print("🏥 NAFLD RISK KATEGORİLERİ")
print("=" * 70)

def classify_nafld_risk(row):
    """
    FIB-4 Index'e göre NAFLD riski sınıflandır
    <1.3: Düşük risk
    1.3-2.67: Orta risk  
    >2.67: Yüksek risk (Advanced Fibrosis)
    """
    fib4 = row['FIB4_Index']
    
    if pd.isna(fib4):
        return 'Unknown'
    elif fib4 < 1.3:
        return 'Low'
    elif fib4 < 2.67:
        return 'Moderate'
    else:
        return 'High'

nafld_clean['NAFLD_Risk'] = nafld_clean.apply(classify_nafld_risk, axis=1)

print(f"\n📊 NAFLD Risk Dağılımı:")
print(nafld_clean['NAFLD_Risk'].value_counts())

print(f"\n📈 Yüzde:")
print(nafld_clean['NAFLD_Risk'].value_counts(normalize=True) * 100)

# Risk grup istatistikleri
print(f"\n" + "=" * 70)
print("📋 Risk Gruplarına Göre İstatistikler:")
print("=" * 70)

for risk in ['Low', 'Moderate', 'High']:
    group = nafld_clean[nafld_clean['NAFLD_Risk'] == risk]
    if len(group) > 0:
        print(f"\n{risk.upper()} RISK ({len(group):,} kişi):")
        print(f"   Age: {group['Age'].mean():.1f} ± {group['Age'].std():.1f}")
        print(f"   BMI: {group['BMI'].mean():.1f} ± {group['BMI'].std():.1f}")
        print(f"   AST: {group['AST'].mean():.1f}")
        print(f"   ALT: {group['ALT'].mean():.1f}")
        print(f"   FIB-4: {group['FIB4_Index'].mean():.2f}")

In [ ]:
# Temiz veriyi kaydet
print("\n" + "=" * 70)
print("💾 VERİLER KAYDEDILIYOR...")

# CSV'ye kaydet
nafld_clean.to_csv('data/processed/nafld_clean.csv', index=False)
print(f"✅ Kaydedildi: data/processed/nafld_clean.csv")
print(f"   Satır: {len(nafld_clean):,}")
print(f"   Sütun: {len(nafld_clean.columns)}")

# GitHub'a push et
import subprocess
result = subprocess.run(
    ['git', 'add', 'data/processed/nafld_clean.csv'],
    capture_output=True
)
result = subprocess.run(
    ['git', 'commit', '-m', 'Add cleaned NAFLD dataset with risk classification'],
    capture_output=True
)
result = subprocess.run(
    ['git', 'push'],
    capture_output=True
)
print(f"✅ GitHub'a push edildi!")

print(f"\n🎉 VERİ HAZIR!")
print(f"   Yetişkin: {len(nafld_clean):,} kişi")
print(f"   NAFLD Risk tahminleri yapıldı")
print(f"   Model training'e hazır!")

In [ ]:
# Klasörü oluştur
import os

os.makedirs('data/processed', exist_ok=True)
print("✅ data/processed/ klasörü oluşturuldu!")

# Şimdi dosya kaydet
nafld_clean.to_csv('data/processed/nafld_clean.csv', index=False)
print(f"✅ Kaydedildi: data/processed/nafld_clean.csv")
print(f"   Satır: {len(nafld_clean):,}")
print(f"   Sütun: {len(nafld_clean.columns)}")

# Klasör kontrol et
print(f"\n📁 Dosyalar:")
import subprocess
result = subprocess.run(['ls', '-lah', 'data/processed/'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# GitHub'a push et
import subprocess

print("🔄 GitHub'a Push Ediliyor...")
print("=" * 70)

# Add
subprocess.run(['git', 'add', 'data/processed/nafld_clean.csv'], 
               capture_output=True)

# Commit
result = subprocess.run(['git', 'commit', '-m', 'Add cleaned NAFLD dataset with risk classification (5550 patients)'], 
                       capture_output=True, text=True)

# Push
result = subprocess.run(['git', 'push'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    print("✅ GitHub'a push edildi!")
else:
    print(f"⚠️  {result.stderr[:100]}")

print("\n🎉 VERİ HAZIR!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import joblib

print("\n" + "=" * 70)
print("🤖 RANDOM FOREST + XGBOOST MODEL TRAINING")
print("=" * 70)

# Feature selection
X_cols = ['Age', 'BMI', 'AST', 'ALT', 'Platelet_K', 'Triglyceride', 
          'Total_Cholesterol', 'LDL', 'HDL', 'SBP', 'DBP', 'Waist_cm']

# Eksik verileri doldur
for col in X_cols:
    nafld_clean[col].fillna(nafld_clean[col].median(), inplace=True)

X = nafld_clean[X_cols]
y = (nafld_clean['NAFLD_Risk'] != 'Low').astype(int)  # 0=Low, 1=Risk

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Train/Test Split:")
print(f"   Train: {len(X_train):,}")
print(f"   Test: {len(X_test):,}")
print(f"   Risk Ratio: {(y == 1).sum():,} / {len(y):,}")

# ========== RANDOM FOREST ==========
print(f"\n" + "-" * 70)
print("🌲 Random Forest Training...")
rf_model = RandomForestClassifier(
    n_estimators=100, 
    random_state=42, 
    n_jobs=-1,
    max_depth=15
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"✅ Random Forest AUC-ROC: {auc_rf:.4f}")

# ========== XGBOOST ==========
print(f"\n" + "-" * 70)
print("🚀 XGBoost Training...")
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

print(f"✅ XGBoost AUC-ROC: {auc_xgb:.4f}")

# ========== KARŞILAŞTIRMA ==========
print(f"\n" + "=" * 70)
print("📊 MODEL KARŞILAŞTIRMASI")
print("=" * 70)
print(f"Random Forest: {auc_rf:.4f}")
print(f"XGBoost:       {auc_xgb:.4f}")
print(f"🏆 En iyi:     {'XGBoost' if auc_xgb > auc_rf else 'Random Forest'}")

# ========== DETAIL REPORT ==========
print(f"\n" + "=" * 70)
print("📋 RANDOM FOREST - Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Low Risk', 'Moderate/High Risk']))

print(f"\n📋 XGBOOST - Classification Report:")
print(classification_report(y_test, y_pred_xgb, target_names=['Low Risk', 'Moderate/High Risk']))

# ========== FEATURE IMPORTANCE ==========
print(f"\n" + "=" * 70)
print("🔍 FEATURE IMPORTANCE")
print("=" * 70)

feature_imp_rf = pd.DataFrame({
    'Feature': X_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nRandom Forest Top 5:")
for idx, row in feature_imp_rf.head(5).iterrows():
    print(f"   {row['Feature']:20} → {row['Importance']:.4f}")

feature_imp_xgb = pd.DataFrame({
    'Feature': X_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\nXGBoost Top 5:")
for idx, row in feature_imp_xgb.head(5).iterrows():
    print(f"   {row['Feature']:20} → {row['Importance']:.4f}")

# ========== SAVE MODELS ==========
import os
os.makedirs('models', exist_ok=True)

joblib.dump(rf_model, 'models/random_forest_nafld.pkl')
joblib.dump(xgb_model, 'models/xgboost_nafld.pkl')
joblib.dump(X_cols, 'models/feature_names.pkl')

print(f"\n✅ Modeller Kaydedildi:")
print(f"   • models/random_forest_nafld.pkl")
print(f"   • models/xgboost_nafld.pkl")
print(f"   • models/feature_names.pkl")

In [ ]:
import os
import joblib

print("\n" + "=" * 70)
print("✅ RANDOM FOREST MODEL EVALUATION")
print("=" * 70)

# Random Forest metric'leri
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf)
recall = recall_score(y_test, y_pred_rf)
f1 = f1_score(y_test, y_pred_rf)

print(f"\n📊 RANDOM FOREST SONUÇLARI:")
print(f"   AUC-ROC:    {auc_rf:.4f} ✨")
print(f"   Precision:  {precision:.4f}")
print(f"   Recall:     {recall:.4f}")
print(f"   F1-Score:   {f1:.4f}")

print(f"\n📋 Confusion Matrix:")
print(f"   TN: {cm[0,0]:,}  FP: {cm[0,1]:,}")
print(f"   FN: {cm[1,0]:,}  TP: {cm[1,1]:,}")

print(f"\n🔍 Feature Importance (Top 8):")
feature_imp = pd.DataFrame({
    'Feature': X_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

for idx, row in feature_imp.head(8).iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"   {row['Feature']:20} {bar} {row['Importance']:.4f}")

# Kaydet
os.makedirs('models', exist_ok=True)
joblib.dump(rf_model, 'models/nafld_model.pkl')
joblib.dump(X_cols, 'models/feature_names.pkl')

print(f"\n✅ Model Kaydedildi:")
print(f"   • models/nafld_model.pkl")
print(f"   • models/feature_names.pkl")

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

print("📈 ROC Curve Çiziliyor...")

fpr, tpr, _ = roc_curve(y_test, y_pred_proba_rf)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(fpr, tpr, label=f'Random Forest (AUC = {auc_rf:.4f})', 
        linewidth=3, color='#2E86AB', marker='o', markersize=3, alpha=0.8)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)

ax.fill_between(fpr, tpr, alpha=0.2, color='#2E86AB')

ax.set_xlabel('False Positive Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('True Positive Rate', fontsize=13, fontweight='bold')
ax.set_title('🏥 NAFLD Risk Prediction - ROC Curve\n(Random Forest Model)', 
             fontsize=15, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])

plt.tight_layout()
plt.savefig('models/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ ROC Curve kaydedildi: models/roc_curve.png")

In [ ]:
import subprocess

print("\n" + "=" * 70)
print("🔄 GitHub'a Push Ediliyor...")
print("=" * 70)

subprocess.run(['git', 'add', 'models/'], capture_output=True)
result = subprocess.run(
    ['git', 'commit', '-m', 'Add Random Forest NAFLD model & ROC curve (AUC: 0.9892)'], 
    capture_output=True, text=True
)
result = subprocess.run(['git', 'push'], capture_output=True, text=True)

print("✅ GitHub'a push edildi!")

print("\n" + "=" * 70)
print("🎉 MODEL TRAINİNG TAMAMLANDI!")
print("=" * 70)
print("\n✨ ÖZET:")
print("   ✅ 5,550 yetişkin NAFLD veri seti")
print("   ✅ Random Forest (AUC: 0.9892)")
print("   ✅ Precision: 94.59%")
print("   ✅ Recall: 89.49%")
print("   ✅ ROC Curve visualization")
print("   ✅ GitHub'da güvenli")